# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madaraf/Starter-Notebooks-Assignment-Flyrank-/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Lane 2 — Refresh / Content Opportunity Scoring** (provisional, revisit by end of Week 4).

I'm picking this lane over the other three for a few concrete reasons:

- It has an **observed label already in the data** (`trend_direction`, and by extension `is_declining_label`), not something I have to invent from scratch. Lanes 1 and 3 (signal analysis, clustering) are more open-ended and don't force me to commit to a decision this early; Lane 4 (CTR/engagement) is a good second choice but is really a special case of the same "which page first?" question this lane already asks more broadly.
- It has a **transparent baseline already built and run** (`scripts/02_baseline_score.py`, committed in `outputs/model_report.md`), so I'm not starting from zero — I can compare any model I train against a real, already-computed number instead of an imagined one.
- It maps cleanly onto a **ranking task with a precision@K metric**, which is the metric type the framing skill flags as the most defensible for "which ones first?" questions, and it's the metric type FlyRank's own product (health scores, priority queues) already uses — so my work is directly comparable to something real.
- The decision it supports is concrete and has a named actor (a content editor with limited review time), which satisfies the "who acts on the output, and what do they do?" test from the framing skill — I struggled to answer that as cleanly for the clustering lane.

I'm not locking this in — Lane 4 (CTR opportunity) is my fallback if, once I dig into the signal audit in Week 4, the refresh label turns out to be too noisy or too easy (precision@50 near the base rate) to be interesting.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Out of thousands of existing content pages, which pages should a content editor review first for refresh, and why?

**Unit of analysis:** one row = one content page (`content_id`), scored using its trailing-90-day search and engagement metrics. Not a client, not a day — a single piece of published content.

**Decision this improves:** the weekly (or sprint-level) prioritization decision an editor already has to make with limited time — which handful of pages, out of thousands, get a human look this week.

**Who acts on the output:** a content editor or SEO strategist working through a fixed-capacity review queue (e.g. can realistically review ~20-50 pages a week). They don't act on the raw score directly — they open the top-ranked pages, read the reason codes (e.g. `declining_with_demand`, `low_ctr_visible_page`), and decide whether to refresh, expand, review CTR/metadata, or leave it monitored.

**The action taken:** refresh the content, expand thin content, review title/meta for CTR, review on-page engagement, or simply monitor (do nothing yet). This is a decision-support ranking, not an automated publishing action — a human always makes the final call, per the repo's data-use rules.

**Cost of a wrong call:**
- *False positive* (page ranked high but wasn't actually worth refreshing): wastes a scarce editor-hour that could have gone to a page that really was declining. With review capacity fixed at ~50/week, every wasted slot has a real opportunity cost.
- *False negative* (a genuinely declining, high-traffic page ranked low or missed): the page keeps losing visibility/clicks silently until someone notices by accident — the exact failure mode ("teams notice too late") this whole lane exists to prevent.
- Because both error types are costly but in different currencies (editor time vs. lost traffic), **precision@K** is the right metric to start with (it directly measures "of the K pages we send an editor to look at, how many were worth it"), with recall as a secondary check once the queue size is fixed.

**Why data/ML helps here (vs. a plain if-statement):** a hand-written rule like "stale AND visible" is easy to write but conflates several different reasons a page could be worth reviewing (declining traffic, weak CTR at a good position, thin content, low engagement) into one bucket. The signals involved — impressions, position, CTR, freshness, word count, engagement — interact in ways that are individually weak but, combined, are learnable: the committed pipeline shows a plain rule getting Precision@50 = 0.240, while a random forest trained on the same signals gets 0.740 (~3x). That gap is exactly the kind of "pattern is real but too tangled for one if-statement" case the framing skill describes as where ML earns its place.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Quick sanity check of the decision framing: how big is the review capacity problem really?
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Total pages in starter slice: {len(df):,}")
print(f"If an editor can review ~50 pages/week, that covers "
      f"{50/len(df):.2%} of the inventory per week — prioritization is not optional, it's forced.")


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three real numbers, pulled from the bundled anonymized starter dataset and the already-run reference pipeline (`scripts/run_all.py`, committed to `outputs/model_report.md`):

1. **The label is common enough to matter, not vanishingly rare.** 16,262 of 30,000 pages (54.2%) carry `is_declining_label = 1`. That's a healthy base rate — not so rare that precision@K is meaningless, not so common that "predict everyone" would already look good.
2. **A learned ranking clearly beats a transparent rule on the same data and metric.** The hand-written baseline rule gets Precision@50 = 0.240 (~12 of the top 50 flagged pages are actually declining); the random forest trained on the same observable features gets Precision@50 = 0.740 (~37 of 50) — roughly a 3x lift, using the *same* client-holdout split and the *same* metric, which is the honest comparison the framing/baseline skills require.
3. **A naive, single-signal heuristic would have failed.** In the Week-1 discovery notebook, `search_volume` correlates with actual `impressions_90d` at only 0.001 — essentially zero. That rules out the simplest possible "lane" (just rank by keyword search volume) and confirms the pattern really is tangled across multiple signals (position, freshness, CTR, engagement, age) rather than something one column already tells you.

Together these three numbers say: there's a real, common-enough target; a simple rule alone leaves a lot on the table; and a slightly less simple model recovers most of it — which is exactly the gap a 7-week capstone can spend its time on (sharper features, better validation, better action mapping) without having to invent the problem from nothing.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json
import pandas as pd

# --- Number 1: how common is the label? ---
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Rows: {len(df):,}")
print(f"Declining-label rate: {df['is_declining_label'].mean():.3f} "
      f"({int(df['is_declining_label'].sum()):,} of {len(df):,} pages)")

# --- Number 2: baseline vs model, same split, same metric ---
# (requires having run `python scripts/run_all.py` at least once, per notebook 01)
try:
    res = json.load(open("outputs/model_results.json"))
    base = res["baseline"]["baseline_precision_at_50"]
    rf = res["models"]["random_forest"]["precision_at_50"]
    print(f"\nBaseline rule  Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
    print(f"Random forest  Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
    print(f"Lift: {rf/base:.1f}x")
except FileNotFoundError:
    print("\noutputs/model_results.json not found yet — run `python scripts/run_all.py` "
          "(or Notebook 01) once, then re-run this cell.")

# --- Number 3: does the obvious single-signal heuristic already work? ---
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"\nCorrelation(search_volume, impressions_90d): {corr:.3f}  "
      "(near zero -> a one-column heuristic isn't enough)")


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim, by the end of this lane:**
- *Observed*: on this anonymized 30k-row snapshot, a ranked model beats a transparent rule at picking pages that were labeled declining, measured by precision@K on a client-holdout split.
- *Decision-support*: "these pages look worth reviewing first, given the evidence" — a ranked queue with reason codes for a human to inspect, not an automatic publishing or editing action.
- *Directional*: which observable signals (freshness, position, CTR, engagement, age) tend to co-occur with the declining label in this dataset, stated with effect sizes and sample sizes, not just "X matters."

**What I will not claim, ever:**
- That refreshing a page **causes** its traffic to recover — this is cross-sectional, non-experimental data; proving causation would need an actual before/after experiment or a causal design I'm not running.
- That I "predicted Google's algorithm," reverse-engineered ranking factors, or found anything about how Search itself works — I modeled outcomes inside one anonymized client portfolio, nothing more.
- That `is_declining_label` (`trend_direction == "down"`) is a perfect measure of "this page needs help" — it's a proxy computed from a 30/60-day impression window, not a validated ground truth, and I'll say so explicitly when I report results.
- Any number from a bucket too small to trust (I'll report n alongside every rate, per the auditing-signals and writing-honest-claims skills), and no accuracy number without its base rate sitting right next to it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Claim ladder for this lane: observed -> directional -> decision-support.")
print("No causal claims without an experiment. No 'predicted Google's algorithm.'")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.